# Data Visualization

> "The simple graph has brought more information to the data analyst's mind than any other device." — John Tukey

A penguin's body mass can range from about 2,700 g to 6,300 g — but **where** do most penguins fall in that range? And do longer flippers go with heavier birds, or lighter ones?

In this lesson we use **seaborn** (built on **matplotlib**) to answer questions like these with plots. We will:

- visualize the **distribution** of a single variable,
- explore **relationships** between two (or more) variables, and
- preview how **faceting** splits a plot into small multiples for fair comparison.

The guiding idea throughout is that **every plot answers a question** — your job is to read the answer off the chart.

**Learning objectives**

- Choose the right plot for categorical vs. numerical variables.
- Build scatterplots step by step and add a line of best fit when a trend matters.
- Compare a numerical variable across groups with box plots and overlapping density curves.
- Summarize two categorical variables with `pd.crosstab` and stacked bar charts.
- Preview faceting with `sns.relplot` as a bridge to the next lesson.


## Setup


In [ ]:
# pandas for tabular data: https://pandas.pydata.org/docs/
import pandas as pd

# matplotlib is the underlying plotting engine: https://matplotlib.org/stable/
import matplotlib.pyplot as plt

# seaborn is a high-level statistical plotting library: https://seaborn.pydata.org/
import seaborn as sns

# Apply seaborn's default theme (colors, fonts, grid) to every matplotlib figure.
sns.set_theme()


## The `penguins` dataset

seaborn ships with a few example datasets. `load_dataset("penguins")` returns the **Palmer Penguins** data as a pandas `DataFrame`: body measurements for penguins on three islands:

![](../assets/lter_penguins.png)

- `species` — the penguin's species (Adelie, Chinstrap, or Gentoo)
- `island` — the island where the penguin was observed
- `bill_length_mm`, `bill_depth_mm` — bill measurements, in millimeters
- `flipper_length_mm` — flipper length, in millimeters
- `body_mass_g` — body mass, in grams
- `sex` — the penguin's sex

A `DataFrame` is a rectangular table: **variables** live in columns and **observations** (here, individual penguins) live in rows.


In [ ]:
# load_dataset downloads (and caches) a tidy example DataFrame.
penguins = sns.load_dataset("penguins")

# head() previews the first 5 rows so we can eyeball the columns and values.
penguins.head()


`info()` is a quick way to see every column, its data type, and how many non-missing values it has. Notice that a few measurement columns have fewer than the total number of rows — those are **missing values**, which seaborn quietly drops when plotting.


In [ ]:
# info() reports column names, dtypes, and non-null counts in one summary.
penguins.info()


# One variable: distributions

How we picture the distribution of a *single* variable depends on whether it is **categorical** or **numerical**.

## A categorical variable

- A variable is **categorical** when it takes one of a small set of values, like `species`.
- A **count plot** (bar chart) shows how many observations fall into each category.


In [ ]:
fig, ax = plt.subplots()

# countplot tallies the rows in each category and draws one bar per category.
sns.countplot(
    data=penguins,
    x="species",
    ax=ax,
)

ax.set_title("Number of penguins per species")


For categories with no natural order, it is often easier to read the chart when the bars are **sorted by frequency**. We compute the order with `value_counts()` (which is already sorted from most to least common) and pass its index to the `order` argument.


In [ ]:
# value_counts() returns counts sorted high-to-low; its index is the species order.
species_by_frequency = penguins["species"].value_counts().index

fig, ax = plt.subplots()

sns.countplot(
    data=penguins,
    x="species",
    order=species_by_frequency,   # draw bars from most to least common
    ax=ax,
)

ax.set_title("Penguins per species, ordered by frequency")


## A numerical variable

A variable is **numerical** when it spans a range of numbers it makes sense to add or average, like `body_mass_g`.

A **histogram** splits the range into equal-width bins and draws a bar whose height is the number of observations in each bin.


In [ ]:
fig, ax = plt.subplots()

# histplot bins a numerical column; binwidth sets the width of each bin (in grams here).
sns.histplot(
    data=penguins,
    x="body_mass_g",
    binwidth=200,   # try other values: too small looks jagged, too large hides the shape
    ax=ax,
)

ax.set_title("Distribution of body mass")


Always try a few bin widths: too narrow makes a jagged chart, too wide hides the shape. A width of 200 g strikes a sensible balance and shows a single peak (unimodal) that trails off to the right (slightly right-skewed).

A **density (KDE) plot** is a smoothed alternative to the histogram. It is useful for continuous data and makes the overall shape — peaks and skew — easy to read at a glance.


In [ ]:
fig, ax = plt.subplots()

# kdeplot draws a smooth density curve; fill shades the area under it.
sns.kdeplot(
    data=penguins,
    x="body_mass_g",
    fill=True,
    ax=ax,
)

ax.set_title("Smoothed density of body mass")


# Relationships between two variables

To visualize a *relationship* we need at least two variables on the plot. Which plot to reach for depends on the **types** of the two variables:

- one numerical + one categorical,
- two categorical, or
- two numerical.

As before, each plot is chosen to answer a specific question.

## A numerical and a categorical variable

> Does body mass differ across penguin species?

A **box plot** summarizes a numerical variable for each category.

- Each box spans the **middle half of the data**
- the _line_ inside marks the **median**
- the _whiskers_ reach the bulk of the rest
- and _isolated points_ flag **possible outliers**


In [ ]:
fig, ax = plt.subplots()

# boxplot: categorical x (species) vs numerical y (body mass), one box per species.
sns.boxplot(
    data=penguins,
    x="species",
    y="body_mass_g",
    ax=ax,
)

ax.set_title("Body mass by species")


The boxes show that Gentoo penguins are clearly heavier than Adelie and Chinstrap, whose distributions overlap a lot.

We can ask the same question with overlapping **density curves**, one per species. Mapping `species` to `hue` draws and colors a curve for each group; `fill` with some transparency makes the overlaps readable.


In [ ]:
fig, ax = plt.subplots()

sns.kdeplot(
    data=penguins,
    x="body_mass_g",
    hue="species",   # one density curve per species, automatically colored
    fill=True,       # shade under each curve
    alpha=0.5,       # transparency so curves don't hide each other
    ax=ax,
)

ax.set_title("Body mass distribution by species")


## Two categorical variables

> How are the species distributed across the islands?

With two categorical variables we count how often each combination occurs. `pd.crosstab` builds a table of counts that we can plot as a **stacked bar chart**.


In [ ]:
# crosstab tallies how many penguins fall in each (island, species) combination.
island_species_counts = pd.crosstab(penguins["island"], penguins["species"])
island_species_counts


In [ ]:
fig, ax = plt.subplots()

# A DataFrame can plot itself; stacked=True piles the species counts within each island.
island_species_counts.plot(
    kind="bar",
    stacked=True,
    ax=ax,
)

ax.set_title("Species counts per island")
ax.set_ylabel("count")


The raw counts are affected by the different number of penguins on each island. To compare the *composition* of each island, convert the counts to **proportions** with `normalize="index"` (each island's bars now sum to 1).


In [ ]:
# normalize="index" turns each row of counts into proportions that sum to 1.
island_species_proportions = pd.crosstab(
    penguins["island"],
    penguins["species"],
    normalize="index",
)

fig, ax = plt.subplots()

island_species_proportions.plot(
    kind="bar",
    stacked=True,
    ax=ax,
)

ax.set_title("Species proportions per island")
ax.set_ylabel("proportion")


Now the story is clear: Gentoo penguins live only on Biscoe, Chinstrap only on Dream, and Adelie are found on all three islands (and are the only species on Torgersen).


## Two numerical variables

> Do penguins with longer flippers weigh more or less than penguins with shorter flippers?

Try to make your answer precise.

- Is the relationship positive or negative?
- Linear or curved?
- Strong or weak?
- Does it depend on the species?

A **scatterplot** of two numerical variables is the natural tool to find out. We map `flipper_length_mm` to the x-axis and `body_mass_g` to the y-axis.


In [ ]:
fig, ax = plt.subplots()

# scatterplot draws one point per penguin. data= is the DataFrame; x/y are column names.
sns.scatterplot(
    data=penguins,
    x="flipper_length_mm",
    y="body_mass_g",
    ax=ax,
)

ax.set_title("Body mass vs. flipper length")


Already we can answer the motivating question:

- the relationship looks **positive** (longer flippers go with heavier penguins)
- **fairly linear** (points cluster around a straight line)
- **moderately strong** (not too much scatter)

It is always wise to be skeptical and ask whether a *third* variable changes the story. Does the relationship differ by species?

We can encode species with color by mapping it to the `hue` aesthetic. seaborn assigns one color per species automatically and adds a legend.


In [ ]:
fig, ax = plt.subplots()

sns.scatterplot(
    data=penguins,
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species",   # color each point by its species, and add a legend
    ax=ax,
)

ax.set_title("Body mass vs. flipper length, by species")


### Adding a line of best fit

To make the trend explicit we can overlay a **regression line** (a straight line fit through the points). `regplot` draws the scatter *and* a best-fit line with a shaded confidence band on the same Axes.


In [ ]:
fig, ax = plt.subplots()

# regplot = scatter points + a fitted straight line (ordinary least squares).
sns.regplot(
    data=penguins,
    x="flipper_length_mm",
    y="body_mass_g",
    ax=ax,
)

ax.set_title("Body mass vs. flipper length with a line of best fit")


The upward-sloping line confirms the positive relationship we saw by eye. The points hug the line reasonably closely, which is the visual signature of a *strong* linear relationship.


# Three or more variables: faceting (preview)

We can squeeze in a third variable by mapping it to `hue`, `style`, or `size` — but adding too many at once makes a plot hard to read.

A cleaner option for a categorical third variable is **faceting**: splitting the data into small side-by-side subplots — the same plot repeated for each subset, on the same scale. seaborn's figure-level `relplot` does this with `col=`.

This is only a preview. In `02_small_multiples.ipynb` you will learn how figure-level functions (`displot`, `relplot`, `catplot`) build faceted grids systematically.


In [ ]:
# relplot is figure-level: it manages its own grid of subplots, so we don't pass ax=.
grid = sns.relplot(
    data=penguins,
    x="flipper_length_mm",
    y="body_mass_g",
    hue="species",   # color points by species
    col="island",      # one subplot (facet) per island
    kind="scatter",
)

grid.figure.suptitle("Body mass vs. flipper length, faceted by island", y=1.02)


## Summary

- For a single **categorical** variable, a **count plot** (`sns.countplot`) shows how many observations fall in each category; sort by frequency for easier reading.
- For a single **numerical** variable, a **histogram** (`sns.histplot`) and a **density plot** (`sns.kdeplot`) reveal the shape of the distribution.
- **Numerical vs. categorical**: a **box plot** (`sns.boxplot`) or overlapping **density curves** (`sns.kdeplot` with `hue`) compare a number across groups.
- **Two categorical**: a **stacked bar chart** of `pd.crosstab` counts shows raw frequencies; switching to **proportions** (`normalize="index"`) compares composition fairly.
- **Two numerical**: a **scatterplot** (`sns.scatterplot`), optionally with a fitted line (`sns.regplot`); map a third variable to `hue` when groups behave differently.
- For a third variable, use `hue`/`style`/`size` sparingly, or **facet** with `sns.relplot(col=...)` — the idea we develop fully in the next lesson.

Next, in `02_small_multiples.ipynb`, you will learn how seaborn's **figure-level** functions turn faceting into a repeatable pattern for comparing distributions and relationships side by side.
